In [1]:
import torch
import torch.nn as nn


def generate_plotneuralnet(model, input_size=(3, 256, 256)):
    layers = list(model.features) + list(model.classifier)

    tex = []
    tex.append(r"\begin{tikzpicture}")
    tex.append(r"\tikzstyle{connection}=[ultra thick,draw=\edgecolor,opacity=0.7]")

    x_offset = 0
    layer_id = 1

    current_channels = input_size[0]
    current_size = input_size[1]

    prev_name = None

    for layer in layers:
        name = f"l{layer_id}"

        if isinstance(layer, nn.Conv2d):
            out_channels = layer.out_channels

            tex.append(
                rf"""
\pic at ({x_offset},0,0) {{RightBandedBox={{
name={name},
caption=Conv,
xlabel={{{{"{out_channels}"}}}},
zlabel={current_size},
fill=\ConvColor,
bandfill=\ConvReluColor,
height={current_size/6},
width={{2}},
depth={current_size/6}
}}}};
"""
            )

            current_channels = out_channels
            prev_name = name
            x_offset += 2

        elif isinstance(layer, nn.MaxPool2d):
            current_size //= 2

            tex.append(
                rf"""
\pic at ({x_offset},0,0) {{Box={{
name={name},
fill=\PoolColor,
opacity=0.5,
height={current_size/6},
width=1,
depth={current_size/6}
}}}};
"""
            )

            prev_name = name
            x_offset += 1.5

        elif isinstance(layer, nn.AdaptiveAvgPool2d):
            current_size = 1

            tex.append(
                rf"""
\pic at ({x_offset},0,0) {{Box={{
name={name},
caption=GAP,
fill=\PoolColor,
height=5,
width=1,
depth=5
}}}};
"""
            )

            prev_name = name
            x_offset += 1.5

        elif isinstance(layer, nn.Linear):
            out_features = layer.out_features

            tex.append(
                rf"""
\pic at ({x_offset},0,0) {{RightBandedBox={{
name={name},
caption=FC,
xlabel={{{{"{out_features}"}}}},
fill=\FcColor,
bandfill=\FcColor,
height=5,
width={{2}},
depth=5
}}}};
"""
            )

            prev_name = name
            x_offset += 2

        layer_id += 1

    tex.append(r"\end{tikzpicture}")

    return "\n".join(tex)


# =====================
# USAGE
# =====================



In [4]:
import os
import sys
sys.path.append(os.path.abspath('..'))


In [5]:
from galaxy_classification import GalaxyCNN

model = GalaxyCNN(num_classes=4)

tikz_code = generate_plotneuralnet(model)

with open("cnn.tex", "w") as f:
    f.write(tikz_code)

print("✅ Generated cnn.tex")

✅ Generated cnn.tex
